# TB Drug Resistance — Exploratory Data Analysis

The system is supported by two datasets: a synthetic patient case base for case-based reasoning and the CRyPTIC cohort of real isolates for resistance classification. This notebook provides an overview of both datasets.

## Synthetic case base

- Outcome signal per feature → similarity weights
- Class imbalance across the six resistance profiles → retrieval coverage
- Trivial baselines to beat → always-predict-success (outcome), modal regimen per profile
- Seed graph coverage and gaps

In [1]:
import ast
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# The notebook runs from the eda/ folder. Source modules live in the sibling
# SRC/ and Evaluation/ folders, and the CRyPTIC tables in Datasets/.

ROOT = Path.cwd().parent
DATA = ROOT / "Datasets"
sys.path[:0] = [str(ROOT / "SRC"), str(ROOT / "Evaluation")]

from cbr_cases import generate_cases
from validation import baseline_accuracy

N_CASES = 1000
SEED = 42
SEVERITY = ["Susceptible", "MonoResistant", "PolyResistant", "MDR", "PreXDR", "XDR"]

frame = pd.DataFrame(generate_cases(N_CASES, SEED))
success = frame["outcome"].eq("success")
len(frame)

1000

## 1. Distribution Sanity Check
Proportions across the entire case base, calculated independently of `CaseGenerator.distribution_summary`, serve as a cross-check to ensure the generation aligns with the targets.

In [2]:
pd.concat({
    "profile": frame["profile"].value_counts(normalize=True).round(3),
    "region": frame["region"].value_counts(normalize=True).round(3),
})

profile  Susceptible        0.500
         MDR                0.180
         MonoResistant      0.120
         PreXDR             0.080
         XDR                0.060
         PolyResistant      0.060
region   SE_Asia            0.277
         African            0.256
         W_Pacific          0.165
         Americas           0.107
         European           0.100
         E_Mediterranean    0.095
Name: proportion, dtype: float64

In [3]:
frame["outcome"].value_counts(normalize=True).round(3)

outcome
success          0.738
ltfu             0.094
death            0.091
failed           0.046
not_evaluated    0.031
Name: proportion, dtype: float64

In [4]:
pd.Series({
    "hiv_positive": frame["hiv_status"].eq("positive").mean(),
    "diabetes": frame["diabetes"].mean(),
    "previous_treatment": frame["previous_treatment"].mean(),
    "mean_age": frame["age"].mean(),
}).round(3)

hiv_positive           0.109
diabetes               0.155
previous_treatment     0.149
mean_age              39.753
dtype: float64

## 2. Class balance across the six resistance profiles
Counts, share, and success rates for each profile, arranged by clinical severity. The dataset is primarily composed of Susceptible cases, while the resistant categories (PolyResistant, PreXDR, XDR) are smaller. This leads to a clear design consideration where neighbor-based retrieval will have abundant data for Susceptible cases but limited data for the less common resistant classes. Consequently, the quality of data per profile is expected to vary rather than remain consistent.

In [5]:
pd.DataFrame({
    "n": frame["profile"].value_counts(),
    "share": frame["profile"].value_counts(normalize=True).round(3),
    "success_rate": success.groupby(frame["profile"]).mean().round(3),
}).reindex(SEVERITY)

,n,share,success_rate
profile,,,
Susceptible,500,0.50,0.822
MonoResistant,120,0.12,0.742
PolyResistant,60,0.06,0.650
MDR,180,0.18,0.717
PreXDR,80,0.08,0.512
XDR,60,0.06,0.483


## 3. Outcome signal by feature
Success rate varies by each candidate similarity feature. Features showing a broad success rate range across their values provide outcome signals and should have a higher weight in the similarity metric. Conversely, features with minimal impact should be weighted less. This empirical approach bases the similarity weights on actual data rather than assumptions. The region is included intentionally: it influences prevalence and demographics in the generator but does not directly affect the outcome. Therefore, a nearly flat region row suggests assigning it a low weight.

In [6]:
frame["age_band"] = pd.cut(frame["age"], bins=[17, 34, 49, 64, 80],
                           labels=["18-34", "35-49", "50-64", "65-80"])


def success_by(column):
    return success.groupby(frame[column], observed=True).mean().round(3)


pd.concat({c: success_by(c) for c in
           ["hiv_status", "previous_treatment", "diabetes", "sex", "age_band", "region"]})

hiv_status          negative           0.753
                    positive           0.615
previous_treatment  False              0.759
                    True               0.617
diabetes            False              0.751
                    True               0.665
sex                 F                  0.801
                    M                  0.710
age_band            18-34              0.739
                    35-49              0.771
                    50-64              0.687
                    65-80              0.675
region              African            0.738
                    Americas           0.757
                    E_Mediterranean    0.674
                    European           0.700
                    SE_Asia            0.762
                    W_Pacific          0.745
Name: outcome, dtype: float64

## 4. Trivial baselines to beat
Two simple predictors establish the minimum standard that a case-based design must surpass. The headline pair is generated by `validation.baseline_accuracy`, the same function used in the validation report. Therefore, the baseline shown here and the baseline CBR tested in `validation.py` are identical by design, not just similar. The per-profile tables below provide context, explaining the source of each headline figure.

**Outcome, always predict success.** Since most cases are successful, predicting success for everyone already yields high accuracy; the success rate per profile equals that baseline's accuracy, and the overall value matches the `outcome` figure from the shared function.

**Regimen, most frequent regimen per profile.** Each profile tends to have a dominant regimen, so this approach scores well. When a profile is dominated by a single regimen (share 1.0), the prediction is trivially correct; in cases where regimens are split across two or three options, there is potential for a more sophisticated model.

In [7]:
records = [{"actual_success": c["outcome"] == "success",
            "profile": c["profile"], "actual_regimen": c["regimen"]}
           for c in frame.to_dict("records")]
baseline_accuracy(records)

{'outcome': 0.738, 'regimen': 0.81}

In [8]:
outcome_baseline = pd.Series({
    "overall": success.mean(),
    **success.groupby(frame["profile"]).mean().to_dict(),
}).round(3)
outcome_baseline.reindex(["overall", *SEVERITY])

overall          0.738
Susceptible      0.822
MonoResistant    0.742
PolyResistant    0.650
MDR              0.717
PreXDR           0.512
XDR              0.483
dtype: float64

In [9]:
def modal_regimen(series):
    return Counter(series).most_common(1)[0][0]


def majority_share(series):
    return series.value_counts(normalize=True).max()


pd.DataFrame({
    "modal_regimen": frame.groupby("profile")["regimen"].agg(modal_regimen),
    "majority_share": frame.groupby("profile")["regimen"].agg(majority_share).round(3),
    "n_regimens": frame.groupby("profile")["regimen"].nunique(),
}).reindex(SEVERITY)

,modal_regimen,majority_share,n_regimens
profile,,,
Susceptible,2HRZE_4HR,1.000,1
MonoResistant,6REZ_Lfx,1.000,1
PolyResistant,Individualized_12mo,0.517,2
MDR,BPaLM,0.450,3
PreXDR,Individualized_18mo,0.650,2
XDR,BPaL,0.433,3


## 5. Seed knowledge-graph composition
This section provides a static overview of the hand-curated strains in `tb_ontology.py`, extracted from the same source blob used by the test suite. This ensures the overview remains database-free and aligned with `test_core`. It details what the curated graph encompasses (profiles, genes, and geography) and, just as importantly, what it omits. These gaps are genuine: queries for absent combinations should return no results rather than be fulfilled by a fabricated strain. The integrity of profile versus mutation relationships is verified in the test suite, not re-established here; this section is meant for exploration, not validation.

In [10]:
tree = ast.parse((ROOT / "SRC" / "tb_ontology.py").read_text())
blobs = {}
for node in ast.walk(tree):
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in ("strains", "strain_data", "mutations"):
                try:
                    blobs[target.id] = ast.literal_eval(node.value)
                except (ValueError, SyntaxError):
                    pass

country = {s["id"]: s["country"] for s in blobs["strains"]}
gene = {m["id"]: m["gene"] for m in blobs["mutations"]}

kg = pd.DataFrame([{
    "strain": r["strain"],
    "country": country.get(r["strain"]),
    "profile": r["profile"],
    "genes": sorted({gene.get(m) for m in r["mutations"]} - {None}),
} for r in blobs["strain_data"]])

kg["profile"].value_counts().reindex(SEVERITY)

profile
Susceptible       6
MonoResistant    22
PolyResistant     9
MDR              10
PreXDR            7
XDR               2
Name: count, dtype: int64

In [11]:
pd.Series({
    "strains": len(kg),
    "countries": kg["country"].nunique(),
    "distinct_genes": len({g for genes in kg["genes"] for g in genes}),
})

strains           56
countries         42
distinct_genes     8
dtype: int64

In [12]:
india = kg[kg["country"] == "India"]
india[["strain", "profile", "genes"]]

,strain,profile,genes
3,TB004,PolyResistant,"[embB, inhA]"
4,TB005,MonoResistant,"[inhA, katG]"
33,TB034,PolyResistant,"[inhA, pncA]"
51,TB052,Susceptible,[]


## 6. CRyPTIC real-world isolates

The remaining sections characterize the CRyPTIC Release 3.4.0 cohort: real *M. tuberculosis* isolates with whole-genome variants graded against the WHO 2023 mutation catalog and measured drug-susceptibility phenotypes. Genotype and phenotype only, with no treatment outcomes.

In [13]:
from feature_engineering import flat, profile, resistant_profile, drug_map

# Reuse the single CRyPTIC profile-derivation defined in feature_engineering so this
# notebook and the model-ready table cannot drift. drug_map mirrors the release
# 3-letter drug codes to system drug names; resistant_profile turns a phenotype
# table into one resistance profile per isolate.
drugs = drug_map(DATA / "DRUG_CODES.csv")

## 6. Data sources and scale
The analysis draws on five tables. EFFECTS grades each isolate's mutations against the WHO 2023 catalog (R/S/U/F); PREDICTIONS gives the catalog's per-drug genotypic call; DST_MEASUREMENTS and UKMYC_PHENOTYPES provide measured susceptibility; DRUG_CODES maps the three-letter drug codes. The genome-wide MUTATIONS table is not used, since EFFECTS provides the same variants in graded form.

In [14]:
for name in ["EFFECTS.parquet", "PREDICTIONS.parquet",
             "DST_MEASUREMENTS.parquet", "UKMYC_PHENOTYPES.parquet"]:
    rows = pq.ParquetFile(DATA / name).metadata.num_rows
    print(f"{name:28s} {rows:>12,} rows")

EFFECTS.parquet                 1,154,127 rows
PREDICTIONS.parquet               810,615 rows
DST_MEASUREMENTS.parquet          660,961 rows
UKMYC_PHENOTYPES.parquet          288,904 rows


## 7. Measured resistance-profile balance
Resistance profile derived from DST measured phenotypes using the system's class definitions. As a resistance-prediction cohort, CRyPTIC carries thousands of isolates in every resistant tier, including pre-XDR and XDR, a markedly different distribution from population prevalence where those tiers would be sparse. The PolyResistant is the smallest class and the narrowest by definition.

In [15]:
dst = resistant_profile(DATA / "DST_MEASUREMENTS.parquet", "PHENOTYPE", drugs)
dst.value_counts().reindex(SEVERITY)

Susceptible      39484
MonoResistant     7707
PolyResistant      874
MDR              10335
PreXDR            4725
XDR               2463
Name: count, dtype: int64

## 8. Cross-assay label concordance
UKMYC is an independent MIC-based assay. Across isolates measured by both methods, the derived profiles are compared. Concordance is high, and the disagreements concentrate in the resistant tiers rather than in Susceptible, consistent with borderline resistance being where assays diverge.

In [16]:
ukmyc = resistant_profile(DATA / "UKMYC_PHENOTYPES.parquet", "BINARY_PHENOTYPE", drugs)
both = pd.DataFrame({"dst": dst, "ukmyc": ukmyc}).dropna()
pd.Series({"measured_by_both": len(both),
           "agreement": round((both["dst"] == both["ukmyc"]).mean(), 3)})

measured_by_both    21568.000
agreement               0.956
dtype: float64

In [17]:
both[both["dst"] != both["ukmyc"]]["dst"].value_counts().reindex(SEVERITY, fill_value=0)

dst
Susceptible        0
MonoResistant    178
PolyResistant     42
MDR              347
PreXDR           192
XDR              198
Name: count, dtype: int64

## 9. Resistance Signal versus Phylogenetic Background
Non-synonymous variants in resistance genes are not always linked to resistance where several common changes (gyrA E21Q, gyrA S95T, katG R463L) serve as phylogenetic markers found in most isolates. To identify true resistance determinants, focus on the catalog's resistance-level isolates. The ranking below, derived from graded variants, highlights well-known resistance mutations such as katG S315T, rpoB S450L, and embB M306V.

In [18]:
def resistance_mutations(path):
    eff = flat(pd.read_parquet(path, columns=["UNIQUEID", "GENE", "MUTATION", "PREDICTION"]), "GENE")
    r = eff[eff["PREDICTION"].astype(str) == "R"]
    gene = r["GENE"].astype("string").fillna("NA")
    mut = gene.str.cat(r["MUTATION"].astype("string").fillna("NA"), sep="_")
    return r.assign(mut=mut).drop_duplicates(["UNIQUEID", "mut"])


resistance = resistance_mutations(DATA / "EFFECTS.parquet")
resistance["mut"].value_counts().head(15)

mut
katG_S315T     17071
rpoB_S450L     12359
rpsL_K43R       8843
embB_M306V      5155
fabG1_c-15t     4833
embB_M306I      3739
gyrA_D94G       2823
rrs_a1401g      2519
rrs_a514c       1652
gyrA_A90V       1643
rpoB_D435V      1564
rpsL_K88R       1451
embB_Q497R      1400
eis_c-12t        784
rrs_c517t        751
Name: count, dtype: Int64

## 10. Genotypic catalog prediction versus measured phenotype
The genotype-based profile from the catalog is compared to the measured phenotype profile, both overall and by category. Agreement is strongest for Susceptible and high-resistance tiers, while it is lowest for PolyResistant and MonoResistant categories, which rely on a small number of borderline calls.

In [19]:
catalog = resistant_profile(DATA / "PREDICTIONS.parquet", "PREDICTION", drugs)
compare = pd.DataFrame({"measured": dst, "catalog": catalog}).dropna()
compare["correct"] = compare["measured"] == compare["catalog"]
pd.Series({"n": len(compare), "exact_agreement": round(compare["correct"].mean(), 3)})

n                  53735.000
exact_agreement        0.813
dtype: float64

In [20]:
compare.groupby("measured")["correct"].mean().reindex(SEVERITY).round(3)

measured
Susceptible      0.891
MonoResistant    0.622
PolyResistant    0.229
MDR              0.743
PreXDR           0.748
XDR              0.785
Name: correct, dtype: float64